In [ ]:
import tensorflow as tf

In [ ]:
from pathlib import Path
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, cache_dir="datasets",extract=True)
text = (Path(path) / "spa-eng" / "spa.txt").read_text()

In [ ]:
import numpy as np
text = text.replace("¡", "").replace("¿", "")
pairs = [line.split("\t") for line in text.splitlines()]
np.random.shuffle(pairs)
sentences_en, sentences_es = zip(*pairs) # separates the pairs into 2 lists

In [ ]:
for i in range(3):
 print(sentences_en[i], "=>", sentences_es[i])


He is talking on the phone. => Él está hablando por teléfono.
It was hot yesterday. => Ayer hacía calor.
He let go of the rope. => Él soltó la cuerda.


In [ ]:
vocab_size = 1000
max_length = 50
text_vec_layer_en = tf.keras.layers.TextVectorization(
 vocab_size, output_sequence_length=max_length)
text_vec_layer_es = tf.keras.layers.TextVectorization(
 vocab_size, output_sequence_length=max_length)
text_vec_layer_en.adapt(sentences_en)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es])

In [ ]:
text_vec_layer_en.get_vocabulary()[:10]

['',
 '[UNK]',
 np.str_('the'),
 np.str_('i'),
 np.str_('to'),
 np.str_('you'),
 np.str_('tom'),
 np.str_('a'),
 np.str_('is'),
 np.str_('he')]

In [ ]:
 text_vec_layer_es.get_vocabulary()[:10]


['',
 '[UNK]',
 np.str_('startofseq'),
 np.str_('endofseq'),
 np.str_('de'),
 np.str_('que'),
 np.str_('a'),
 np.str_('no'),
 np.str_('tom'),
 np.str_('la')]

In [ ]:
X_train = tf.constant(sentences_en[:100_000])
X_valid = tf.constant(sentences_en[100_000:])
X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:100_000]])
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[100_000:]])
Y_train = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[:100_000]])
Y_valid = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[100_000:]])

In [ ]:
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

In [ ]:
embed_size = 128
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size,
 mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size,
 mask_zero=True)
encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

In [ ]:
encoder = tf.keras.layers.LSTM(512, return_state=True)
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

In [ ]:
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

In [ ]:
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(decoder_outputs)

In [ ]:
model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs],outputs=[Y_proba])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
model.fit((X_train, X_train_dec), Y_train, epochs=10, validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 79s 23ms/step - accuracy: 0.0515 - loss: 3.4994 - val_accuracy: 0.0747 - val_loss: 2.1458
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 72s 23ms/step - accuracy: 0.0795 - loss: 1.9242 - val_accuracy: 0.0875 - val_loss: 1.6173
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 71s 23ms/step - accuracy: 0.0922 - loss: 1.4299 - val_accuracy: 0.0930 - val_loss: 1.4071
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 71s 23ms/step - accuracy: 0.0998 - loss: 1.1648 - val_accuracy: 0.0955 - val_loss: 1.3179
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 72s 23ms/step - accuracy: 0.1047 - loss: 0.9888 - val_accuracy: 0.0967 - val_loss: 1.2829
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 72s 23ms/step - accuracy: 0.1092 - loss: 0.8479 - val_accuracy: 0.0972 - val_loss: 1.2769
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 76s 24ms/step - accuracy: 0.1131 - loss: 0.7311 - val_accuracy: 0.0971 - val_loss: 1.2913
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 71s 23ms/step - accuracy: 0.1166 -

In [ ]:
def translate(sentence_en):
 translation = ""
 for word_idx in range(max_length):
    X = tf.constant([sentence_en], dtype=tf.string) # encoder input
    X_dec = tf.constant(["startofseq " + translation], dtype=tf.string) # decoder input
    y_proba = model.predict((X, X_dec))[0, word_idx] # last token's probas
    predicted_word_id = np.argmax(y_proba)
    predicted_word = text_vec_layer_es.get_vocabulary()[predicted_word_id]
    if predicted_word == "endofseq":
      break
    translation += " " + predicted_word
 return translation.strip()

In [ ]:
translate("I like to play")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


'me gusta jugar'